In [1]:
import os
import json
from getpass import getpass

import weaviate
from weaviate.auth import AuthApiKey
from weaviate.classes.query import Filter, MetadataQuery

In [ ]:
if "WEAVIATE_DEMO_API_KEY" not in os.environ:
    os.environ["WEAVIATE_DEMO_API_KEY"] = getpass("WEAVIATE_DEMO_API_KEY: ")
if "COHERE_API_KEY" not in os.environ:
    os.environ["COHERE_API_KEY"] = getpass("COHERE_API_KEY: ")

In [ ]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url="https://cohere-demo.weaviate.network/",
    auth_credentials=AuthApiKey(os.environ["WEAVIATE_DEMO_API_KEY"]),
    headers={"X-Cohere-Api-Key": os.environ["COHERE_API_KEY"]},
)
print(client.is_ready())

In [ ]:
articles = client.collections.use("Articles")
count = articles.aggregate.over_all(total_count=True)
print("objects:", count.total_count)

In [ ]:
response = articles.query.near_text(
    query="Vacation spots in California",
    limit=5,
    return_properties=["text", "title", "url", "views", "lang"],
    return_metadata=MetadataQuery(distance=True),
)
for obj in response.objects:
    print(obj.metadata.distance)
    print(obj.properties.get("title"), obj.properties.get("lang"), obj.properties.get("url"))

In [ ]:
client.close()